# Load from bronze table

In [0]:
import pyspark.sql.functions as F

cards_df        = spark.table("jrvs_databricks_fundamentals.bronze.cards_data")
transactions_df = spark.table("jrvs_databricks_fundamentals.bronze.transactions_data")
users_df        = spark.table("jrvs_databricks_fundamentals.bronze.users_data")
mcc_df          = spark.table("jrvs_databricks_fundamentals.bronze.mcc_codes")
fraud_df        = spark.table("jrvs_databricks_fundamentals.bronze.fraud_labels")

## Transformation
# keep all rows for the most part just clean each data and remove the null aspect to it

### Keep all rows regardless of null columns

In [ ]:

silver_users_df = (
    users_df
    .withColumnRenamed("id", "user_id")
    .filter(F.col("user_id").isNotNull())
    .dropDuplicates(["user_id"])
)

silver_cards_df = (
    cards_df
    .withColumnRenamed("id", "card_id")
    .withColumnRenamed("client_id", "user_id")   # FK -> users
    .withColumn(
        "card_on_dark_web",
        F.when(F.upper(F.trim(F.col("card_on_dark_web"))) == "YES", True)
         .when(F.upper(F.trim(F.col("card_on_dark_web"))) == "NO", False)
         .otherwise(None)
    )
    .filter(F.col("card_id").isNotNull())
    .dropDuplicates(["card_id"])
)


silver_transactions_df = (
    transactions_df
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("client_id", "user_id")   # FK -> users
    .withColumnRenamed("mcc", "mcc_code")        # FK -> mcc_codes
    # zip came in as double  -> zip format has to be 5 digits; double -> int -> string(5 digits)
    .withColumn(
        "zip",
        F.when(F.col("zip").isNull(), None)
         .otherwise(F.format_string("%05d", F.col("zip").cast("int")))
    )
    .filter(F.col("transaction_id").isNotNull())   
    .dropDuplicates(["transaction_id"])
)

In [ ]:
import pyspark.sql.functions as F

silver_mcc_df = (
    mcc_df
    .filter(F.col("mcc_code").isNotNull())  #PK
    .dropDuplicates(["mcc_code"])
)

## Fraud
silver_fraud_df = (
    fraud_df
    .withColumnRenamed("TransactionID", "transaction_id")
    .withColumnRenamed("Value", "is_fraud")
    .withColumn("transaction_id", F.col("transaction_id").cast("int"))
    .withColumn(
        "is_fraud",
        F.when(F.upper(F.trim(F.col("is_fraud"))) == "YES", True)
         .when(F.upper(F.trim(F.col("is_fraud"))) == "NO", False)
         .otherwise(None)
    )
    .filter(F.col("transaction_id").isNotNull() & F.col("is_fraud").isNotNull())
    .dropDuplicates(["transaction_id"])
)


# Save as Table

In [ ]:
silver_users_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("jrvs_databricks_fundamentals.silver.users_data")

silver_cards_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("jrvs_databricks_fundamentals.silver.cards_data")

silver_transactions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("jrvs_databricks_fundamentals.silver.transactions_data")

silver_mcc_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("jrvs_databricks_fundamentals.silver.mcc_codes")

silver_fraud_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("jrvs_databricks_fundamentals.silver.fraud_labels")